<a href="https://colab.research.google.com/github/Uditgohar17/From-Decentralized-Prosperity-to-Centralized-Fragility-2021/blob/main/From_Decentralized_Prosperity_to_Centralized_Fragility_A_Data_Science_Study_of_India%E2%80%99s_Economic_%26_Governance_Transition%22_from_2010_to_2022.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# PROJECT: From Decentralized Prosperity to Centralized Fragility
# PURPOSE: End-to-End Data Science Pipeline for BCA Final Year Project
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. SETUP & LIBRARIES
# ------------------------------------------------------------------------------
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from scipy import stats
import warnings

# Suppress warnings for clean output
warnings.filterwarnings('ignore')
np.random.seed(42) # Ensures results are reproducible

print("✅ PHASE 0: Environment Setup Complete.")

# ------------------------------------------------------------------------------
# 2. PHASE 1: DATA INGESTION (Synthetic Generation)
# ------------------------------------------------------------------------------
# We simulate a dataset representing 15 Indian states over 14 years (2010-2023).
# This replaces the need for external CSV uploads for your demo.

def generate_project_data():
    years = np.arange(2010, 2024)
    # Categorizing states to create realistic variance
    rich_states = ['Maharashtra', 'Tamil Nadu', 'Karnataka', 'Gujarat', 'Telangana']
    emerging_states = ['Rajasthan', 'Madhya Pradesh', 'Uttar Pradesh', 'West Bengal', 'Andhra Pradesh']
    hill_ne_states = ['Himachal Pradesh', 'Uttarakhand', 'Assam', 'Meghalaya', 'Tripura']

    data = []

    for year in years:
        # Define the "Centralization Era" (Post-2017 GST/Policy shift)
        era = 'Post-Centralization' if year >= 2017 else 'Pre-Centralization'

        for group, states in zip([rich_states, emerging_states, hill_ne_states],
                                 ['High', 'Medium', 'Low']):
            for state in states:
                # Base Logic: Economic Scale
                if group == rich_states:
                    base_gsdp = np.random.uniform(12000, 20000)
                    tax_efficiency = 0.7  # High own tax collection
                elif group == emerging_states:
                    base_gsdp = np.random.uniform(6000, 11000)
                    tax_efficiency = 0.5
                else:
                    base_gsdp = np.random.uniform(1500, 4000)
                    tax_efficiency = 0.3 # Low own tax, needs central support

                # Growth Trend (with a dip in 2020 for COVID)
                growth = np.random.uniform(0.05, 0.09)
                shock = -0.06 if year == 2020 else 0

                # Apply growth over time logic
                current_gsdp = base_gsdp * ((1 + growth + shock) ** (year - 2010))

                # --- THE CENTRALIZATION SHIFT LOGIC ---
                # Post-2017, States lose some ability to collect own tax (GST effect)
                # and rely more on central transfers.
                if year >= 2017:
                    own_tax_ratio = np.random.uniform(0.30, 0.45) * tax_efficiency
                    central_transfer_ratio = np.random.uniform(0.30, 0.50) # Higher dependency
                else:
                    own_tax_ratio = np.random.uniform(0.40, 0.60) * tax_efficiency
                    central_transfer_ratio = np.random.uniform(0.20, 0.35) # Lower dependency

                # Calculating absolutes based on GSDP
                total_revenue = current_gsdp * 0.15
                own_tax = total_revenue * own_tax_ratio
                central_transfer = total_revenue * central_transfer_ratio

                # Governance Index (Randomized but sticky per state)
                gov_score = np.random.uniform(4, 9)

                data.append({
                    'State': state,
                    'Year': year,
                    'Category': group,
                    'Era': era,
                    'GSDP_Crore': round(current_gsdp, 2),
                    'Own_Tax_Revenue': round(own_tax, 2),
                    'Central_Transfers': round(central_transfer, 2),
                    'Total_Expenditure': round(total_revenue * 1.1, 2), # Spend > Earn (Deficit)
                    'Governance_Index': round(gov_score, 2)
                })

    return pd.DataFrame(data)

df = generate_project_data()
print(f"✅ PHASE 1: Data Ingestion Complete. Dataset shape: {df.shape}")

# ------------------------------------------------------------------------------
# 3. PHASE 2: PREPROCESSING & FEATURE ENGINEERING
# ------------------------------------------------------------------------------
# Creating the KPI metrics defined in your problem statement.

# 1. Fiscal Autonomy: Ability to self-fund.
df['Fiscal_Autonomy_Ratio'] = (df['Own_Tax_Revenue'] / df['Total_Expenditure']) * 100

# 2. Central Dependency: Reliance on New Delhi.
df['Central_Dependency_Ratio'] = (df['Central_Transfers'] / df['Total_Expenditure']) * 100

# 3. Deficit Ratio: How much they overspend.
df['Fiscal_Deficit_Ratio'] = ((df['Total_Expenditure'] - (df['Own_Tax_Revenue'] + df['Central_Transfers'])) / df['GSDP_Crore']) * 100

print("✅ PHASE 2: Feature Engineering Complete (Autonomy & Dependency Ratios created).")

# ------------------------------------------------------------------------------
# 4. PHASE 3: EXPLORATORY DATA ANALYSIS (EDA) & STATS
# ------------------------------------------------------------------------------
# A. T-Test to prove the "Fragility" hypothesis
pre_autonomy = df[df['Era'] == 'Pre-Centralization']['Fiscal_Autonomy_Ratio']
post_autonomy = df[df['Era'] == 'Post-Centralization']['Fiscal_Autonomy_Ratio']

t_stat, p_val = stats.ttest_ind(pre_autonomy, post_autonomy, equal_var=False)
stat_result = "Significant Decline" if p_val < 0.05 else "No Significant Change"

print(f"\n--- Statistical Hypothesis Test ---")
print(f"Hypothesis: Did Fiscal Autonomy drop post-2017?")
print(f"T-Statistic: {t_stat:.4f} | P-Value: {p_val:.4e}")
print(f"Conclusion: {stat_result} in Autonomy confirmed.\n")

# ------------------------------------------------------------------------------
# 5. PHASE 4: MACHINE LEARNING MODELING
# ------------------------------------------------------------------------------

# --- A. UNSUPERVISED: K-MEANS CLUSTERING (Segmentation) ---
# Grouping states into 'Resilient', 'Dependent', 'Struggling'
X_cluster = df.groupby('State')[['Fiscal_Autonomy_Ratio', 'GSDP_Crore', 'Governance_Index']].mean()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

kmeans = KMeans(n_clusters=3, random_state=42)
X_cluster['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# Map numeric cluster to meaningful labels (Rule-based mapping for demo stability)
# Sorting clusters by Autonomy to label them correctly
cluster_means = X_cluster.groupby('Cluster_ID')['Fiscal_Autonomy_Ratio'].mean().sort_values()
label_map = {
    cluster_means.index[0]: 'Fragile/Dependent',
    cluster_means.index[1]: 'Emerging',
    cluster_means.index[2]: 'Resilient/Autonomous'
}
X_cluster['Cluster_Label'] = X_cluster['Cluster_ID'].map(label_map)

# Merge back to main dataframe
df = df.merge(X_cluster[['Cluster_Label']], on='State', how='left')

# --- B. SUPERVISED: RANDOM FOREST (Driver Analysis) ---
# Predicting GSDP based on fiscal/governance inputs
features = ['Fiscal_Autonomy_Ratio', 'Central_Dependency_Ratio', 'Governance_Index', 'Total_Expenditure']
target = 'GSDP_Crore'

X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
r2 = r2_score(y_test, y_pred)

print(f"✅ PHASE 4: ML Modeling Complete.")
print(f"   > Segmentation: States clustered into {X_cluster['Cluster_Label'].unique()}")
print(f"   > Prediction: Random Forest R2 Score = {r2:.3f}")

# ------------------------------------------------------------------------------
# 6. PHASE 5: INTERACTIVE DASHBOARD
# ------------------------------------------------------------------------------
# Building the final UI for your presentation

def build_dashboard():
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "1. The Shift: Autonomy vs Dependency (2010-2023)",
            "2. State Segmentation (ML Clusters)",
            "3. Feature Importance (What drives Growth?)",
            "4. Autonomy Heatmap by State"
        ),
        specs=[[{"type": "xy"}, {"type": "xy"}],
               [{"type": "xy"}, {"type": "xy"}]]
    )

    # Chart 1: Line Chart (Time Series)
    yearly_avg = df.groupby('Year')[['Fiscal_Autonomy_Ratio', 'Central_Dependency_Ratio']].mean().reset_index()
    fig.add_trace(go.Scatter(x=yearly_avg['Year'], y=yearly_avg['Fiscal_Autonomy_Ratio'],
                             mode='lines+markers', name='Fiscal Autonomy %', line=dict(color='green')), row=1, col=1)
    fig.add_trace(go.Scatter(x=yearly_avg['Year'], y=yearly_avg['Central_Dependency_Ratio'],
                             mode='lines+markers', name='Central Dependency %', line=dict(color='red')), row=1, col=1)
    fig.add_vline(x=2017, line_dash="dash", line_color="gray", annotation_text="Policy Shift", row=1, col=1)

    # Chart 2: Scatter Plot (Clusters)
    # We plot the aggregated cluster data
    for label in X_cluster['Cluster_Label'].unique():
        subset = X_cluster[X_cluster['Cluster_Label'] == label]
        fig.add_trace(go.Scatter(x=subset['Fiscal_Autonomy_Ratio'], y=subset['GSDP_Crore'],
                                 mode='markers+text', text=subset.index, textposition="top center",
                                 name=label, marker=dict(size=10)), row=1, col=2)

    # Chart 3: Bar Chart (Feature Importance)
    importances = pd.DataFrame({'Feature': features, 'Importance': rf_model.feature_importances_})
    importances = importances.sort_values(by='Importance', ascending=True)
    fig.add_trace(go.Bar(x=importances['Importance'], y=importances['Feature'], orientation='h',
                         marker=dict(color='teal'), name='Impact'), row=2, col=1)

    # Chart 4: Heatmap (State Performance)
    # Using the latest year data for a snapshot
    latest_year = df[df['Year'] == 2023]
    fig.add_trace(go.Heatmap(z=latest_year['Fiscal_Autonomy_Ratio'],
                             x=latest_year['Era'], # Dummy x just for visual
                             y=latest_year['State'],
                             colorscale='RdYlGn', showscale=True, name='Autonomy'), row=2, col=2)

    # Layout Updates
    fig.update_layout(height=900, width=1200,
                      title_text="Project Dashboard: From Decentralized Prosperity to Centralized Fragility",
                      showlegend=True, template="plotly_white")

    fig.show()

print("✅ PHASE 5: Dashboard Generated Below.")
build_dashboard()

✅ PHASE 0: Environment Setup Complete.
✅ PHASE 1: Data Ingestion Complete. Dataset shape: (182, 9)
✅ PHASE 2: Feature Engineering Complete (Autonomy & Dependency Ratios created).

--- Statistical Hypothesis Test ---
Hypothesis: Did Fiscal Autonomy drop post-2017?
T-Statistic: 6.4894 | P-Value: 9.4344e-10
Conclusion: Significant Decline in Autonomy confirmed.

✅ PHASE 4: ML Modeling Complete.
   > Segmentation: States clustered into ['Resilient/Autonomous' 'Fragile/Dependent' 'Emerging']
   > Prediction: Random Forest R2 Score = 0.995
✅ PHASE 5: Dashboard Generated Below.


In [2]:
df['Fiscal_Autonomy_Ratio'] = (df['Own_Tax_Revenue'] / df['Total_Expenditure']) * 100
df['Central_Dependency_Ratio'] = (df['Central_Transfers'] / df['Total_Expenditure']) * 100
df['Fiscal_Deficit_Ratio'] = ((df['Total_Expenditure'] - (df['Own_Tax_Revenue'] + df['Central_Transfers'])) / df['GSDP_Crore']) * 100

print("✅ PHASE 2: Feature Engineering Complete (Autonomy & Dependency Ratios created).")

✅ PHASE 2: Feature Engineering Complete (Autonomy & Dependency Ratios created).


In [3]:
yearly_avg = df.groupby('Year')[['Fiscal_Autonomy_Ratio', 'Central_Dependency_Ratio']].mean().reset_index()

fig = px.line(yearly_avg, x='Year', y=['Fiscal_Autonomy_Ratio', 'Central_Dependency_Ratio'],
              title='Time-Series Trend: Fiscal Autonomy vs. Central Dependency (2010-2023)',
              labels={'value': 'Percentage (%)', 'Year': 'Year'},
              color_discrete_map={'Fiscal_Autonomy_Ratio': 'green', 'Central_Dependency_Ratio': 'red'})

fig.add_vline(x=2017, line_dash="dash", line_color="gray", annotation_text="Policy Shift")
fig.update_layout(hovermode="x unified")
fig.show()

print("✅ Instruction 1: Time-Series Trends Visualized.")

✅ Instruction 1: Time-Series Trends Visualized.


In [4]:
print(f"--- Statistical Hypothesis Test ---")
print(f"Hypothesis: Did Fiscal Autonomy drop post-2017?")
print(f"T-Statistic: {t_stat:.4f} | P-Value: {p_val:.4e}")
print(f"Conclusion: {stat_result} in Autonomy confirmed.\n")
print("✅ Instruction 2: Statistical Hypothesis Test Results Interpreted.")

--- Statistical Hypothesis Test ---
Hypothesis: Did Fiscal Autonomy drop post-2017?
T-Statistic: 6.4894 | P-Value: 9.4344e-10
Conclusion: Significant Decline in Autonomy confirmed.

✅ Instruction 2: Statistical Hypothesis Test Results Interpreted.


In [12]:
fig_autonomy = px.box(df, x='Category', y='Fiscal_Autonomy_Ratio', color='Category',
                      title='Fiscal Autonomy Ratio Distribution by State Category',
                      labels={'Fiscal_Autonomy_Ratio': 'Fiscal Autonomy (%)', 'Category': 'State Category'})
fig_autonomy.show()

fig_dependency = px.box(df, x='Category', y='Central_Dependency_Ratio', color='Category',
                        title='Central Dependency Ratio Distribution by State Category',
                        labels={'Central_Dependency_Ratio': 'Central Dependency (%)', 'Category': 'State Category'})
fig_dependency.show()

fig_gsdp = px.box(df, x='Category', y='GSDP_Crore', color='Category',
                  title='GSDP Distribution by State Category',
                  labels={'GSDP_Crore': 'GSDP (Crore)', 'Category': 'State Category'})
fig_gsdp.show()

print("✅ Instruction 3: Variable Distributions by State Category Explored.")

✅ Instruction 3: Variable Distributions by State Category Explored.


In [11]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy import stats

# Re-defining generate_project_data to fix the 'Category' column issue
def generate_project_data():
    years = np.arange(2010, 2024)
    # Categorizing states to create realistic variance
    rich_states = ['Maharashtra', 'Tamil Nadu', 'Karnataka', 'Gujarat', 'Telangana']
    emerging_states = ['Rajasthan', 'Madhya Pradesh', 'Uttar Pradesh', 'West Bengal', 'Andhra Pradesh']
    hill_ne_states = ['Himachal Pradesh', 'Uttarakhand', 'Assam', 'Meghalaya', 'Tripura']

    data = []

    # Corrected zip order: category_label is the string, states_list is the list of states
    for category_label, states_list in zip(['High', 'Medium', 'Low'],
                                           [rich_states, emerging_states, hill_ne_states]):
        for year in years:
            # Define the "Centralization Era" (Post-2017 GST/Policy shift)
            era = 'Post-Centralization' if year >= 2017 else 'Pre-Centralization'

            for state in states_list:
                # Base Logic: Economic Scale
                if category_label == 'High':
                    base_gsdp = np.random.uniform(12000, 20000)
                    tax_efficiency = 0.7  # High own tax collection
                elif category_label == 'Medium':
                    base_gsdp = np.random.uniform(6000, 11000)
                    tax_efficiency = 0.5
                else: # category_label == 'Low'
                    base_gsdp = np.random.uniform(1500, 4000)
                    tax_efficiency = 0.3 # Low own tax, needs central support

                # Growth Trend (with a dip in 2020 for COVID)
                growth = np.random.uniform(0.05, 0.09)
                shock = -0.06 if year == 2020 else 0

                # Apply growth over time logic
                current_gsdp = base_gsdp * ((1 + growth + shock) ** (year - 2010))

                # --- THE CENTRALIZATION SHIFT LOGIC ---
                # Post-2017, States lose some ability to collect own tax (GST effect)
                # and rely more on central transfers.
                if year >= 2017:
                    own_tax_ratio = np.random.uniform(0.30, 0.45) * tax_efficiency
                    central_transfer_ratio = np.random.uniform(0.30, 0.50) # Higher dependency
                else:
                    own_tax_ratio = np.random.uniform(0.40, 0.60) * tax_efficiency
                    central_transfer_ratio = np.random.uniform(0.20, 0.35) # Lower dependency

                # Calculating absolutes based on GSDP
                total_revenue = current_gsdp * 0.15
                own_tax = total_revenue * own_tax_ratio
                central_transfer = total_revenue * central_transfer_ratio

                # Governance Index (Randomized but sticky per state)
                gov_score = np.random.uniform(4, 9)

                data.append({
                    'State': state,
                    'Year': year,
                    'Category': category_label, # Assign the string label here
                    'Era': era,
                    'GSDP_Crore': round(current_gsdp, 2),
                    'Own_Tax_Revenue': round(own_tax, 2),
                    'Central_Transfers': round(central_transfer, 2),
                    'Total_Expenditure': round(total_revenue * 1.1, 2), # Spend > Earn (Deficit)
                    'Governance_Index': round(gov_score, 2)
                })

    return pd.DataFrame(data)

# Re-generate the DataFrame with the corrected categories
df = generate_project_data()
print(f"✅ PHASE 1: Data Ingestion Complete (Category column fixed). Dataset shape: {df.shape}")

# ------------------------------------------------------------------------------
# 3. PHASE 2: PREPROCESSING & FEATURE ENGINEERING (Re-run)
# ------------------------------------------------------------------------------
# Creating the KPI metrics defined in your problem statement.

# 1. Fiscal Autonomy: Ability to self-fund.
df['Fiscal_Autonomy_Ratio'] = (df['Own_Tax_Revenue'] / df['Total_Expenditure']) * 100

# 2. Central Dependency: Reliance on New Delhi.
df['Central_Dependency_Ratio'] = (df['Central_Transfers'] / df['Total_Expenditure']) * 100

# 3. Deficit Ratio: How much they overspend.
df['Fiscal_Deficit_Ratio'] = ((df['Total_Expenditure'] - (df['Own_Tax_Revenue'] + df['Central_Transfers'])) / df['GSDP_Crore']) * 100

print("✅ PHASE 2: Feature Engineering Complete (Autonomy & Dependency Ratios re-created).")

# ------------------------------------------------------------------------------
# 4. PHASE 3: EXPLORATORY DATA ANALYSIS (EDA) & STATS (Re-run relevant parts)
# ------------------------------------------------------------------------------
# A. T-Test to prove the "Fragility" hypothesis
pre_autonomy = df[df['Era'] == 'Pre-Centralization']['Fiscal_Autonomy_Ratio']
post_autonomy = df[df['Era'] == 'Post-Centralization']['Fiscal_Autonomy_Ratio']

t_stat, p_val = stats.ttest_ind(pre_autonomy, post_autonomy, equal_var=False)
stat_result = "Significant Decline" if p_val < 0.05 else "No Significant Change"

print(f"\n--- Statistical Hypothesis Test ---")
print(f"Hypothesis: Did Fiscal Autonomy drop post-2017?")
print(f"T-Statistic: {t_stat:.4f} | P-Value: {p_val:.4e}")
print(f"Conclusion: {stat_result} in Autonomy confirmed.\n")
print("✅ Statistical Hypothesis Test (Re-run).")


# ------------------------------------------------------------------------------
# 5. PHASE 4: MACHINE LEARNING MODELING (Re-run relevant parts)
# ------------------------------------------------------------------------------

# --- A. UNSUPERVISED: K-MEANS CLUSTERING (Segmentation) ---
# Grouping states into 'Resilient', 'Dependent', 'Struggling'
X_cluster = df.groupby('State')[['Fiscal_Autonomy_Ratio', 'GSDP_Crore', 'Governance_Index']].mean()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

kmeans = KMeans(n_clusters=3, random_state=42)
X_cluster['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# Map numeric cluster to meaningful labels (Rule-based mapping for demo stability)
# Sorting clusters by Autonomy to label them correctly
cluster_means = X_cluster.groupby('Cluster_ID')['Fiscal_Autonomy_Ratio'].mean().sort_values()
label_map = {
    cluster_means.index[0]: 'Fragile/Dependent',
    cluster_means.index[1]: 'Emerging',
    cluster_means.index[2]: 'Resilient/Autonomous'
}
X_cluster['Cluster_Label'] = X_cluster['Cluster_ID'].map(label_map)

# Merge back to main dataframe
df = df.merge(X_cluster[['Cluster_Label']], on='State', how='left')
print("✅ ML Clustering (Re-run).")

# Now re-run the plotting code that previously failed
fig_autonomy = px.box(df, x='Category', y='Fiscal_Autonomy_Ratio', color='Category',
                      title='Fiscal Autonomy Ratio Distribution by State Category',
                      labels={'Fiscal_Autonomy_Ratio': 'Fiscal Autonomy (%)', 'Category': 'State Category'})
fig_autonomy.show()

fig_dependency = px.box(df, x='Category', y='Central_Dependency_Ratio', color='Category',
                        title='Central Dependency Ratio Distribution by State Category',
                        labels={'Central_Dependency_Ratio': 'Central Dependency (%)', 'Category': 'State Category'})
fig_dependency.show()

fig_gsdp = px.box(df, x='Category', y='GSDP_Crore', color='Category',
                  title='GSDP Distribution by State Category',
                  labels={'GSDP_Crore': 'GSDP (Crore)', 'Category': 'State Category'})
fig_gsdp.show()

print("✅ Instruction 3: Variable Distributions by State Category Explored (Fixed and Visualized).")

✅ PHASE 1: Data Ingestion Complete (Category column fixed). Dataset shape: (210, 9)
✅ PHASE 2: Feature Engineering Complete (Autonomy & Dependency Ratios re-created).

--- Statistical Hypothesis Test ---
Hypothesis: Did Fiscal Autonomy drop post-2017?
T-Statistic: 6.0200 | P-Value: 9.0197e-09
Conclusion: Significant Decline in Autonomy confirmed.

✅ Statistical Hypothesis Test (Re-run).
✅ ML Clustering (Re-run).


✅ Instruction 3: Variable Distributions by State Category Explored (Fixed and Visualized).


In [10]:
cluster_characteristics = X_cluster.groupby('Cluster_Label')[['Fiscal_Autonomy_Ratio', 'GSDP_Crore', 'Governance_Index']].mean()
print("\n--- Cluster Characteristics ---")
print(cluster_characteristics)
print("\n✅ Instruction 1 & 2: Cluster characteristics calculated and displayed.")


--- Cluster Characteristics ---
                      Fiscal_Autonomy_Ratio    GSDP_Crore  Governance_Index
Cluster_Label                                                              
Emerging                          20.157597  14345.973393          6.396071
Fragile/Dependent                 12.101755   4229.965714          6.203571
Resilient/Autonomous              28.041869  24920.535000          5.999286

✅ Instruction 1 & 2: Cluster characteristics calculated and displayed.


In [9]:
print(f"Random Forest Regressor R2 Score: {r2:.3f}")
print("✅ Instruction 1: R2 Score printed.")

Random Forest Regressor R2 Score: 0.995
✅ Instruction 1: R2 Score printed.


In [8]:
importances = pd.DataFrame({'Feature': features, 'Importance': rf_model.feature_importances_})
importances = importances.sort_values(by='Importance', ascending=False)
print("\n--- Feature Importances (Top Drivers of GSDP) ---")
print(importances)
print("\n✅ Instructions 2, 3 & 4: Feature importances extracted, sorted, and printed.")


--- Feature Importances (Top Drivers of GSDP) ---
                    Feature  Importance
3         Total_Expenditure    0.990045
2          Governance_Index    0.007144
1  Central_Dependency_Ratio    0.002232
0     Fiscal_Autonomy_Ratio    0.000580

✅ Instructions 2, 3 & 4: Feature importances extracted, sorted, and printed.


In [7]:
build_dashboard()
print("✅ Dashboard generated and displayed.")

✅ Dashboard generated and displayed.
